# 02 - Feature Relevance Correlation

This notebook checks whether historical signals are related to future traffic/pressure.



In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Detect the project root whether this notebook runs from the root folder or notebooks/.
current_working_directory_path = Path.cwd()

if (current_working_directory_path / "data/analytical").exists():
    project_root_path = current_working_directory_path
else:
    project_root_path = current_working_directory_path.parent

# The correlation analysis starts from hourly analytical data, not raw data.
business_load_hourly_file_path = project_root_path / "data/analytical/business_load_hourly.parquet"

## 1. Load Hourly Analytical Data

In [2]:
# Read hourly area-level business data.
business_load_hourly = pd.read_parquet(business_load_hourly_file_path)

# Sort by area and time so lag/lead operations are correct.
business_load_hourly = business_load_hourly.sort_values(["area", "window_start"]).reset_index(drop=True)

# Display the basic shape and date range.
pd.DataFrame([{
    "rows": len(business_load_hourly),
    "areas": business_load_hourly["area"].nunique(),
    "min_window_start": business_load_hourly["window_start"].min(),
    "max_window_start": business_load_hourly["window_start"].max(),
}])

,rows,areas,min_window_start,max_window_start
0,1887726,2073,2022-01-01 00:00:00+00:00,2026-05-19 12:00:00+00:00


## 2. Create Future Targets

These are the values we want to predict before they happen.

In [3]:
# Create a copy so the original dataframe stays unchanged.
correlation_dataset = business_load_hourly.copy()

# Future business traffic target: orders one hour ahead.
correlation_dataset["orders_count_next_1h"] = (
    correlation_dataset.groupby("area")["orders_count"].shift(-1)
)

# Future driver pressure target: driver requests one hour ahead.
correlation_dataset["driver_requests_count_next_1h"] = (
    correlation_dataset.groupby("area")["driver_requests_count"].shift(-1)
)

# High traffic is defined using the global 90th percentile of observed hourly order counts.
# This is a simple first definition. It can be changed later by business needs.
high_traffic_threshold = correlation_dataset["orders_count"].quantile(0.90)

correlation_dataset["high_traffic_next_1h"] = (
    correlation_dataset["orders_count_next_1h"] >= high_traffic_threshold
).astype("float")

pd.DataFrame([{"high_traffic_threshold": high_traffic_threshold}])

,high_traffic_threshold
0,5.0


## 3. Create Historical Candidate Features

Only lagged values are used here. That protects us from target leakage.

In [4]:
# These are candidate signals that should exist in the hourly analytical dataset.
candidate_signal_names = [
    "orders_count",
    "driver_requests_count",
    "requests_per_order",
    "rejection_rate",
    "acceptance_rate",
    "cancellation_rate",
]

# These lag windows represent recently observed history.
candidate_lag_hours = [1, 2, 3, 6, 12, 24, 168]

# Create lagged candidate features area by area.
for signal_name in candidate_signal_names:
    if signal_name not in correlation_dataset.columns:
        continue
    
    for lag_hour in candidate_lag_hours:
        lagged_feature_name = f"{signal_name}_lag_{lag_hour}h"
        correlation_dataset[lagged_feature_name] = (
            correlation_dataset.groupby("area")[signal_name].shift(lag_hour)
        )

## 4. Correlation Table

This table is a first signal, not a final feature-selection decision.

Tree models can use nonlinear relationships, so low linear correlation does not always mean useless. But this table helps us detect obvious strong/weak candidates.

In [5]:
# Select only lagged candidate features.
lagged_candidate_feature_names = [
    column_name
    for column_name in correlation_dataset.columns
    if "_lag_" in column_name
]

# These are the future targets we compare against.
future_target_names = [
    "orders_count_next_1h",
    "driver_requests_count_next_1h",
    "high_traffic_next_1h",
]

correlation_rows = []

for feature_name in lagged_candidate_feature_names:
    for target_name in future_target_names:
        analysis_frame = correlation_dataset[[feature_name, target_name]].dropna()
        
        if len(analysis_frame) == 0:
            continue
        
        correlation_rows.append({
            "feature_name": feature_name,
            "target_name": target_name,
            "pearson_correlation": analysis_frame[feature_name].corr(analysis_frame[target_name], method="pearson"),
            "spearman_correlation": analysis_frame[feature_name].corr(analysis_frame[target_name], method="spearman"),
            "rows_used": len(analysis_frame),
        })

feature_correlation_table = pd.DataFrame(correlation_rows)

feature_correlation_table = feature_correlation_table.assign(
    absolute_spearman_correlation=lambda dataframe: dataframe["spearman_correlation"].abs()
).sort_values("absolute_spearman_correlation", ascending=False)

feature_correlation_table

,feature_name,target_name,pearson_correlation,spearman_correlation,rows_used,absolute_spearman_correlation
0,orders_count_lag_1h,orders_count_next_1h,0.462575,0.398893,1884321,0.398893
2,orders_count_lag_1h,high_traffic_next_1h,0.391970,0.345050,1885653,0.345050
3,orders_count_lag_2h,orders_count_next_1h,0.376365,0.334984,1883205,0.334984
12,orders_count_lag_12h,orders_count_next_1h,0.337332,0.333221,1875186,0.333221
5,orders_count_lag_2h,high_traffic_next_1h,0.321723,0.289298,1884321,0.289298
...,...,...,...,...,...,...
99,acceptance_rate_lag_24h,orders_count_next_1h,-0.046140,-0.031732,1799287,0.031732
101,acceptance_rate_lag_24h,high_traffic_next_1h,-0.038184,-0.031227,1799811,0.031227
62,requests_per_order_lag_168h,high_traffic_next_1h,-0.004784,0.031027,1807277,0.031027
102,acceptance_rate_lag_168h,orders_count_next_1h,-0.044162,-0.029686,1744615,0.029686
